In [1]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD.parquet"
data: pd.DataFrame = read_data.read_asset(nombre_activo)

In [2]:
# Se espera una string en el siguiente formato
# año-mes
def next_date(current_date: str) -> str:
    age: int = int(current_date[0:4])
    month: int = int(current_date[-2:])
    if month == 12:
        return f"{age + 1}-01"

    if month < 9:
        return f"{age}-0{month + 1}" 
    return f"{age}-{month + 1}"

def get_first_days(df: pd.DataFrame, days: int = 10) -> pd.DataFrame:
    inicio_real = df.index[0].normalize()

    fin_semana = (inicio_real + pd.Timedelta(days=days)).strftime('%Y-%m-%d')

    return df.loc[:fin_semana]

In [3]:
first_date = data.index[0].strftime('%Y-%m')
last_date = data.index[-1].strftime('%Y-%m')

print(f"inicia en {first_date} y termina en {last_date}")

inicia en 2025-02 y termina en 2026-02


In [ ]:
import find_best
import keys
import pandas as pd
from use_tecnics import simple_methods

keys.methods = simple_methods

ventana_entrenamiento = pd.Timedelta(weeks=4)
ventana_testeo = pd.Timedelta(weeks=1)

inicio_train = data.index[0].normalize()
fin_datos = data.index[-1]

while True:
    fin_train = inicio_train + ventana_entrenamiento
    fin_test = fin_train + ventana_testeo

    if fin_train >= fin_datos:
        break

    train_data = data.loc[inicio_train : fin_train - pd.Timedelta(minutes=1)]

    test_data = data.loc[fin_train : fin_test - pd.Timedelta(minutes=1)]

    if test_data.empty:
        break

    print(f"Entrenando MA (4 semanas): {inicio_train.strftime('%Y-%m-%d')} a {fin_train.strftime('%Y-%m-%d')}")

    mans: list = find_best.opti_main(train_data)

    print(f"Resultados del testeo FUERA DE MUESTRA (1 semana)")
    print(f"Operando desde: {test_data.index[0].strftime('%Y-%m-%d %H:%M')}")
    print(f"Hasta: {test_data.index[-1].strftime('%Y-%m-%d %H:%M')}")
    
    find_best.read_results(mans, test_data)

    inicio_train = inicio_train + ventana_testeo

Entrenando MA (4 semanas): 2025-02-21 a 2025-03-21
Resultado obtenido entrenando desde 2025-02-21 hasta 2025-03-20
Método: TEMA, Datos optimizados [np.int64(2), np.int64(43)]

hit ratio: 0.5954504906333631
risk reward: 0.8687956992978302
profit factor: 1.3331520213363255
trades: 2242
Resultados del testeo FUERA DE MUESTRA (1 semana)
Operando desde: 2025-03-21 00:01
Hasta: 2025-03-27 23:58
Rendimiento de una ma con:
método: TEMA, vela: 2, añadidos: [np.int64(43)]
hit ratio: 0.6076923076923076
risk reward: 0.616269389448846
profit factor: 0.9835410457870472
trades: 520


Entrenando MA (4 semanas): 2025-02-28 a 2025-03-28
